##  Trader Behavior vs Market Sentiment Analysis

## Objective
Analyze how market sentiment (Fear vs Greed) impacts trader performance and behavior.

Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# display settings
pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

## Data Loading

We load both datasets and perform initial inspection to understand structure, data types, and potential quality issues.

In [2]:
# Load datasets
sentiment_df = pd.read_csv('../data/raw/fear_greed_index.csv')
trades_df = pd.read_csv('../data/raw/historical_data.csv')

# Preview
print("Sentiment Data:")
display(sentiment_df.head())

print("\nTrades Data:")
display(trades_df.head())

Sentiment Data:


,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05



Trades Data:


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [3]:
# Shape
print("Sentiment shape:", sentiment_df.shape)
print("Trades shape:", trades_df.shape)

# Columns
print("\nSentiment columns:", sentiment_df.columns)
print("\nTrades columns:", trades_df.columns)

# Data types
print("\nSentiment dtypes:")
print(sentiment_df.dtypes)

print("\nTrades dtypes:")
print(trades_df.dtypes)

Sentiment shape: (2644, 4)
Trades shape: (211224, 16)

Sentiment columns: Index(['timestamp', 'value', 'classification', 'date'], dtype='object')

Trades columns: Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

Sentiment dtypes:
timestamp          int64
value              int64
classification    object
date              object
dtype: object

Trades dtypes:
Account              object
Coin                 object
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                 object
Timestamp IST        object
Start Position      float64
Direction            object
Closed PnL          float64
Transaction Hash     object
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timesta

In [4]:
# Missing values
print("Sentiment missing values:\n", sentiment_df.isnull().sum())
print("\nTrades missing values:\n", trades_df.isnull().sum())

Sentiment missing values:
 timestamp         0
value             0
classification    0
date              0
dtype: int64

Trades missing values:
 Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64


In [5]:
print("Sentiment duplicates:", sentiment_df.duplicated().sum())
print("Trades duplicates:", trades_df.duplicated().sum())

Sentiment duplicates: 0
Trades duplicates: 0


In [6]:
# Numerical summary
print(trades_df.describe())

       Execution Price   Size Tokens      Size USD  Start Position  \
count    211224.000000  2.112240e+05  2.112240e+05    2.112240e+05   
mean      11414.723350  4.623365e+03  5.639451e+03   -2.994625e+04   
std       29447.654868  1.042729e+05  3.657514e+04    6.738074e+05   
min           0.000005  8.740000e-07  0.000000e+00   -1.433463e+07   
25%           4.854700  2.940000e+00  1.937900e+02   -3.762311e+02   
50%          18.280000  3.200000e+01  5.970450e+02    8.472793e+01   
75%         101.580000  1.879025e+02  2.058960e+03    9.337278e+03   
max      109004.000000  1.582244e+07  3.921431e+06    3.050948e+07   

          Closed PnL      Order ID            Fee      Trade ID     Timestamp  
count  211224.000000  2.112240e+05  211224.000000  2.112240e+05  2.112240e+05  
mean       48.749001  6.965388e+10       1.163967  5.628549e+14  1.737744e+12  
std       919.164828  1.835753e+10       6.758854  3.257565e+14  8.689920e+09  
min   -117990.104100  1.732711e+08      -1.175712

## Initial Observations & Questions

- How does trader PnL vary across sentiment (Fear vs Greed)?
- Do traders take higher leverage during Greed periods?
- Is trade frequency higher during Fear (panic trading)?
- Are losses more extreme in one sentiment regime?
- Do traders change long/short bias based on sentiment?

## Column Understanding

### Trades Dataset

- account → unique trader ID
- closedPnL → profit/loss per trade
- size → trade size (position size)
- leverage → risk multiplier
- side → BUY/SELL (used for long/short ratio)
- time → timestamp of trade
- symbol → traded asset
- event → type of trade action

### Sentiment Dataset

- date → day-level timestamp
- classification → Fear / Greed (market sentiment)

In [9]:
print(trades_df.columns.tolist())
print(sentiment_df.columns.tolist())

['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']
['timestamp', 'value', 'classification', 'date']


In [10]:
# Standardize column names (lowercase + underscore)
trades_df.columns = trades_df.columns.str.lower().str.replace(" ", "_")
sentiment_df.columns = sentiment_df.columns.str.lower().str.replace(" ", "_")

print(trades_df.columns)
print(sentiment_df.columns)

Index(['account', 'coin', 'execution_price', 'size_tokens', 'size_usd', 'side',
       'timestamp_ist', 'start_position', 'direction', 'closed_pnl',
       'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id',
       'timestamp'],
      dtype='object')
Index(['timestamp', 'value', 'classification', 'date'], dtype='object')


In [12]:
trades_df['timestamp_ist'] = pd.to_datetime(
    trades_df['timestamp_ist'],
    dayfirst=True
)

In [13]:
trades_df['date'] = trades_df['timestamp_ist'].dt.date

In [14]:
trades_df[['timestamp_ist', 'date']].head()

,timestamp_ist,date
0,2024-12-02 22:50:00,2024-12-02
1,2024-12-02 22:50:00,2024-12-02
2,2024-12-02 22:50:00,2024-12-02
3,2024-12-02 22:50:00,2024-12-02
4,2024-12-02 22:50:00,2024-12-02


In [15]:
trades_df['timestamp_ist'].isnull().sum()

np.int64(0)

In [16]:
# Extract date
trades_df['date'] = trades_df['timestamp_ist'].dt.date

In [17]:
sentiment_df['date'] = pd.to_datetime(sentiment_df['date']).dt.date

In [18]:
trades_df.isnull().sum()

account             0
coin                0
execution_price     0
size_tokens         0
size_usd            0
side                0
timestamp_ist       0
start_position      0
direction           0
closed_pnl          0
transaction_hash    0
order_id            0
crossed             0
fee                 0
trade_id            0
timestamp           0
date                0
dtype: int64

In [19]:
trades_df = trades_df.dropna(subset=['closed_pnl', 'account'])

In [20]:
trades_df = trades_df.drop_duplicates()
sentiment_df = sentiment_df.drop_duplicates()

In [21]:
trades_df = trades_df[
    ['account', 'size_usd', 'direction', 'closed_pnl', 'fee', 'date']
]

In [22]:
merged_df = trades_df.merge(
    sentiment_df[['date', 'classification']],
    on='date',
    how='inner'
)

In [23]:
print(merged_df.head())
print(merged_df['classification'].value_counts())

                                      account  size_usd direction  closed_pnl  \
0  0xae5eacaf9c6b9111fd53034a602c192a04e082ed   7872.16       Buy         0.0   
1  0xae5eacaf9c6b9111fd53034a602c192a04e082ed    127.68       Buy         0.0   
2  0xae5eacaf9c6b9111fd53034a602c192a04e082ed   1150.63       Buy         0.0   
3  0xae5eacaf9c6b9111fd53034a602c192a04e082ed   1142.04       Buy         0.0   
4  0xae5eacaf9c6b9111fd53034a602c192a04e082ed     69.75       Buy         0.0   

        fee        date classification  
0  0.345404  2024-12-02  Extreme Greed  
1  0.005600  2024-12-02  Extreme Greed  
2  0.050431  2024-12-02  Extreme Greed  
3  0.050043  2024-12-02  Extreme Greed  
4  0.003055  2024-12-02  Extreme Greed  
classification
Fear             61837
Greed            50303
Extreme Greed    39992
Neutral          37686
Extreme Fear     21400
Name: count, dtype: int64


## Feature Engineering

We aggregate transaction-level data into daily trader-level metrics to capture performance, consistency and behavioral patterns.

These features allow us to compare how trader behavior varies across different market sentiment regimes.

In [24]:
daily_pnl = (
    merged_df
    .groupby(['account', 'date', 'classification'])['closed_pnl']
    .sum()
    .reset_index()
)

In [25]:
merged_df['is_profit'] = merged_df['closed_pnl'] > 0

win_rate = (
    merged_df
    .groupby(['account', 'date', 'classification'])['is_profit']
    .mean()
    .reset_index(name='win_rate')
)

In [26]:
avg_size = (
    merged_df
    .groupby(['account', 'date', 'classification'])['size_usd']
    .mean()
    .reset_index(name='avg_trade_size')
)

In [27]:
trades_per_day = (
    merged_df
    .groupby(['account', 'date', 'classification'])
    .size()
    .reset_index(name='trades_count')
)

In [28]:
merged_df['is_long'] = merged_df['direction'].str.lower() == 'long'

long_short = (
    merged_df
    .groupby(['account', 'date', 'classification'])['is_long']
    .mean()
    .reset_index(name='long_ratio')
)

In [29]:
final_df = daily_pnl \
    .merge(win_rate, on=['account', 'date', 'classification']) \
    .merge(avg_size, on=['account', 'date', 'classification']) \
    .merge(trades_per_day, on=['account', 'date', 'classification']) \
    .merge(long_short, on=['account', 'date', 'classification'])

In [30]:
final_df.head()

,account,date,classification,closed_pnl,win_rate,avg_trade_size,trades_count,long_ratio
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,Extreme Greed,0.0,0.000000,5089.718249,177,0.0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,Extreme Greed,0.0,0.000000,7976.664412,68,0.0
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,Extreme Greed,0.0,0.000000,23734.500000,40,0.0
3,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-22,Extreme Greed,-21227.0,0.000000,28186.666667,12,0.0
4,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-26,Extreme Greed,1603.1,0.444444,17248.148148,27,0.0
